# DisloCluster — post-process an existing run

Re-render figures, movies, discrete loops and TEM slices from a run directory
that already exists. Nothing here re-solves anything: every cell reads
`march_state.npz` and `evl_coupled/`.

Point `RUN_DIR` at the run you want, or leave it as `None` to take the newest.

In [ ]:
from pathlib import Path

from dislocluster_code import paths
from dislocluster_code.post import march_report, discrete_loops, loop_movie, tem_slices

# None -> the most recent run. `find_runs` searches every output root, so runs
# made before Simulations/output/ existed -- the ones still under
# ZrMicro/output/ -- are found too.
RUN_DIR = None

print("available runs (oldest first):")
for r in paths.find_runs():
    print(f"   {r.parent.parent.name}/{r.parent.name}/{r.name}")

RUN_DIR = Path(RUN_DIR) if RUN_DIR else paths.latest_run()
if RUN_DIR is None:
    raise SystemExit(f"no march runs under {[str(d) for d in paths.OUTPUT_DIRS]}")
print()
print("using:", RUN_DIR)

In [ ]:
import json

cfg_file = RUN_DIR / "config.json"
if cfg_file.is_file():
    cfg = json.loads(cfg_file.read_text())
    print(json.dumps(cfg["derived"], indent=2))
else:
    print("(no config.json — this run predates the Simulations notebooks)")

## Field panels, grain-boundary profiles, volume averages

In [ ]:
N_DOSES  = 6
MOVIES   = False

argv = [str(RUN_DIR), "--n-doses", str(N_DOSES)]
if not MOVIES:
    argv.append("--no-movies")
march_report.main(argv)

## Discrete loops

Samples the continuum loop field into individual loops: counts
`N_k = Σ_j n_k(x_j) V_j` with Voronoi volumes, positions them as an
inhomogeneous Poisson sample so the spatial structure survives, and sets each
radius by conserving the defect count.

This is what replaced the earlier `loops_*` overlays, which drew a size map as
if it were an ensemble — at 1e-4 dpa the old figure drew 2500 platelets into a
domain physically containing 2.7 ⟨a⟩₁ loops.

In [ ]:
discrete_loops.main([str(RUN_DIR)])

## TEM-style projected slices and loop movies

In [ ]:
tem_slices.main([str(RUN_DIR)])
# loop_movie.main([str(RUN_DIR), "--fps", "5"])    # needs ffmpeg

## The report

In [ ]:
from IPython.display import Markdown, display

rep = RUN_DIR / "report.md"
display(Markdown(rep.read_text(encoding="utf-8")) if rep.is_file()
        else Markdown("*no report.md yet — run the cell above*"))